# 04 — Nearside Perspective (Satellite View)

The `NearsidePerspective` projection places the camera at a finite altitude above Earth,
producing the realistic curved horizon and foreshortening you see in geostationary imagery.

---
**Stack used:** `noawclg`, `xarray`, `numpy`, `matplotlib`, `cartopy`, `cmocean`

In [ ]:
import noawclg
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import copy

In [ ]:
BG = "#0d1117"; LAND = "#13171d"; OCEAN = "#0d1520"
COAST = "#2d6db5"; BORDER = "#2a2f36"
TEXT = "#e6edf3"; SUBTEXT = "#8b949e"
BOXBG = "#161b22"; BOXEDGE = "#30363d"

plt.rcParams.update({
    "figure.facecolor":  BG,
    "axes.facecolor":    BG,
    "savefig.facecolor": BG,
    "text.color":        TEXT,
    "font.family":       "monospace",
})

## 1. Satellite altitude reference

| Height (m) | Description |
|---|---|
| 200 000 | Low Earth Orbit — very tight view |
| 3 000 000 | Regional — South America fills the disk |
| **35 786 000** | **Geostationary (GOES / Meteosat / Himawari)** |
| 100 000 000 | Near-orthographic — almost no perspective distortion |

In [ ]:
# Sub-satellite point — longitude and latitude directly below the camera
SAT_LON    = -75.0          # GOES-East approximate longitude
SAT_LAT    =   0.0          # equatorial orbit
SAT_HEIGHT = 35_786_000     # geostationary altitude in metres

## 2. Load data

In [ ]:
ds = noawclg.load(
    var="cape",
    run_date="20260418",
    cycle="00",
    forecast_hours=[0],
)

lat   = ds["lat"].values
lon   = ds["lon"].values
field = ds["cape"].isel(time=0).values   # J/kg, already usable

# 0–360 → −180/180
lon_180 = np.where(lon > 180, lon - 360, lon)
order   = np.argsort(lon_180)
lon_180 = lon_180[order]
field   = field[:, order]

date_str = str(ds["cape"].time.values[0])[:10]

## 3. Build the nearside axes

`NearsidePerspective` produces a circular disk.  
Any data outside the visible hemisphere is automatically clipped by cartopy.

In [ ]:
levels_cape = np.array([100, 250, 500, 750, 1000, 1500, 2000, 2500,
                         3000, 4000, 5000, 6000])
cmap_c = copy.copy(plt.cm.YlOrRd)
cmap_c.set_under(alpha=0); cmap_c.set_bad(alpha=0)
norm_c = mcolors.BoundaryNorm(levels_cape, ncolors=cmap_c.N, clip=False)

proj = ccrs.NearsidePerspective(
    central_longitude=SAT_LON,
    central_latitude=SAT_LAT,
    satellite_height=SAT_HEIGHT,
)

fig, ax = plt.subplots(
    figsize=(10, 10),
    subplot_kw={"projection": proj},
    facecolor=BG,
)
ax.set_facecolor(BG)
ax.set_global()

ax.add_feature(cfeature.OCEAN.with_scale("110m"),  facecolor=OCEAN, zorder=0)
ax.add_feature(cfeature.LAND.with_scale("110m"),   facecolor=LAND,  zorder=0)
ax.add_feature(cfeature.COASTLINE.with_scale("110m"),
               edgecolor=COAST, linewidth=0.6, zorder=3)
ax.add_feature(cfeature.BORDERS.with_scale("110m"),
               edgecolor=BORDER, linewidth=0.4, zorder=3)

cf = ax.pcolormesh(
    lon_180, lat, field,
    cmap=cmap_c, norm=norm_c,
    transform=ccrs.PlateCarree(),
    zorder=1,
)

cb = fig.colorbar(cf, ax=ax, orientation="horizontal",
                  pad=0.03, fraction=0.03, shrink=0.85)
cb.set_label("CAPE (J/kg)", fontsize=8, color=SUBTEXT)
cb.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb.outline.set_edgecolor(BOXEDGE)

ax.set_title("CAPE", fontsize=10, fontweight="bold", color=TEXT, loc="left", pad=6)
ax.set_title(f"GFS 00z  {date_str}  |  GOES-East view",
             fontsize=7, color=SUBTEXT, loc="right", pad=6)

fig.tight_layout()
fig.savefig("nearside_cape.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Compare different satellite heights

In [ ]:
heights = [
    (3_000_000,  "Regional (3 000 km)"),
    (35_786_000, "Geostationary (35 786 km)"),
    (100_000_000, "Near-orthographic (100 000 km)"),
]

fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 7),
    subplot_kw={"projection": None},
    facecolor=BG,
)
plt.close(fig)  # will rebuild with individual projections

fig2 = plt.figure(figsize=(18, 7), facecolor=BG)

for i, (height, label) in enumerate(heights):
    proj_h = ccrs.NearsidePerspective(
        central_longitude=SAT_LON,
        central_latitude=SAT_LAT,
        satellite_height=height,
    )
    ax_h = fig2.add_subplot(1, 3, i + 1, projection=proj_h, facecolor=BG)
    ax_h.set_global()
    ax_h.add_feature(cfeature.OCEAN.with_scale("110m"), facecolor=OCEAN, zorder=0)
    ax_h.add_feature(cfeature.LAND.with_scale("110m"),  facecolor=LAND,  zorder=0)
    ax_h.add_feature(cfeature.COASTLINE.with_scale("110m"),
                     edgecolor=COAST, linewidth=0.5, zorder=3)
    ax_h.pcolormesh(lon_180, lat, field,
                    cmap=cmap_c, norm=norm_c,
                    transform=ccrs.PlateCarree(), zorder=1)
    ax_h.set_title(label, fontsize=8, color=TEXT)

fig2.suptitle("Effect of satellite height on NearsidePerspective",
              fontsize=10, color=TEXT, y=1.01)
fig2.tight_layout()
fig2.savefig("nearside_heights_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. Exercises

1. **Change the sub-satellite point** — try `SAT_LON=0, SAT_LAT=0` (Meteosat / Africa-centred) or `SAT_LON=140, SAT_LAT=0` (Himawari / Pacific).
2. **Inclined orbit** — try `SAT_LAT=10` or `SAT_LAT=-15` to tilt the visible disk.
3. **Different variable** — load `"pwat"` (precipitable water) and choose `cmocean.cm.haline`.
4. **Build an animation** — load `forecast_hours=range(0, 121, 3)`, loop over time steps, save each frame as a PNG, then encode with `imageio.get_writer()`.

---
**Next:** [05_ocean_data.ipynb](05_ocean_data.ipynb)